## GPU ENGINE
[Rapids AI open beta Polars post](https://developer.nvidia.com/blog/polars-gpu-engine-powered-by-rapids-cudf-now-available-in-open-beta)

[Google Colab notebook demo from Rapids AI](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/accelerated_data_processing_examples/polars_gpu_engine_demo.ipynb?utm_source=nvidia+mktg&utm_medium=web&utm_campaign=polars+launch)

In [1]:
from IPython import get_ipython
import polars as pl
import os
zone_schema = {"LocationID": pl.Int32, "Borough": pl.String, "Zone": pl.String, "service_zone": pl.String}

df_zones = pl.scan_csv("data/nyc-taxi/taxi_zone_lookup.csv", schema=zone_schema)

df = (
    pl.scan_parquet("data/nyc-taxi/yellow.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)
df_green = (
    pl.scan_parquet("data/nyc-taxi/green.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)

In [ ]:
df.head().collect(engine="streaming")

## Streaming Engine

In [ ]:
%%time
df_green.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine="streaming")

## GPU in memory
Using the in-memory is the fastest than using managed memory (UVM streaming) but the memory is limited to GPU memory (as it Polars eager).

In [12]:
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "0"  # Forcing in-memory engine to not use UVM, just for the demo
gpu_in_mem_engine = pl.GPUEngine(executor="in-memory", raise_on_fail=True)

In [ ]:
%%time
# This will fail if GPU memory is not large enough
df_green.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine=gpu_in_mem_engine)

In [ ]:
result, prof = (
    df_green.group_by("trip_distance")
    .agg(
        pl.len().alias("trip_count"),
        pl.sum("fare_amount").alias("total_revenue"),
        pl.mean("fare_amount").alias("avg_fare"),
        pl.mean("trip_distance").alias("avg_distance"),
        pl.mean("tip_amount").alias("avg_tip"),
    )
    .sort("trip_distance")
    .head()
    .profile(engine=gpu_in_mem_engine)
)
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))

## GPU in memory. More complex queries

In [ ]:
# Recommended to reset kernel to free GPU correctly because some part of the dataset is cached in memory from previous execution
# in the notebook and import again polars import and df definition at the top of the notebook
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
%%time
df_green.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine="streaming")

In [ ]:
%%time
df_green.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=gpu_in_mem_engine)

In [ ]:
result, prof = (
    df_green.group_by(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"])
    .agg(
        pl.len().alias("trip_count"),
        pl.sum("fare_amount").alias("total_revenue"),
        pl.mean("fare_amount").alias("avg_fare"),
        pl.mean("trip_distance").alias("avg_distance"),
        pl.mean("tip_amount").alias("avg_tip"),
    )
    .sort(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"])
    .head()
    .profile(engine=gpu_in_mem_engine)
)
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))